In [0]:
import os
import re
import json
from datetime import datetime

# ---- CONFIG ----
VOLUME_ROOT = "/Volumes/xliidw_dev_lpl/application/adhoc_files/scanner utility/Xli_Wins/XLI_WINS_Extract1" # root of your file dump in UC Volumes

# --- MODIFIED: Define a LIST of strings you want to search for ---
# Add all the strings you want to find in this list.
SEARCH_STRINGS = [
    "conf_scd_shared",
    "initialize_abc_logger_objects",
    "ingest_distribute_adhoc_job_trigger",
    "ingest_distribute_job_group_trigger_miu.py",
    "ingest_distribute_job_group_trigger_1.py" # Example of a sensitive string
]

# File extensions worth scanning (notebooks exported as .py/.sql/.ipynb, ADF as .json, etc.)
SCAN_EXTENSIONS = {".py", ".sql", ".json", ".ipynb", ".txt", ".scala", ".r", ".xml", ".html"}

# ---- SCAN ----
# The results will now be a list of dictionaries, each representing a file with findings
# For example:
# [
#   {
#     "file": "/path/to/file1.py",
#     "found_strings": {
#       "string_A": [{"line": 10, "match": "..."}],
#       "string_B": [{"line": 25, "match": "..."}]
#     }
#   },
#   ...
# ]
results = []

# --- MODIFIED: Function to find multiple strings in content ---
def find_multiple_string_occurrences(content, strings_to_find):
    all_hits_for_file = {}
    for search_str in strings_to_find:
        # Escape the string to treat special regex characters literally
        search_pattern_regex = re.escape(search_str)
        hits_for_this_string = []
        for m in re.finditer(search_pattern_regex, content, re.IGNORECASE):
            line_no = content.count("\n", 0, m.start()) + 1
            # Store the line number and the matched text (truncated for display)
            hits_for_this_string.append({"line": line_no, "match": m.group(0)[:120]})
        if hits_for_this_string:
            all_hits_for_file[search_str] = hits_for_this_string
    return all_hits_for_file

file_count = 0
error_count = 0

def scan_file(full_path):
    """Scan a single file for the search strings."""
    global file_count, error_count
    ext = os.path.splitext(full_path)[1].lower()
    if ext not in SCAN_EXTENSIONS:
        return
    file_count += 1
    try:
        with open(full_path, "r", errors="ignore") as f:
            content = f.read()
    except Exception as e:
        error_count += 1
        return
    file_found_strings = find_multiple_string_occurrences(content, SEARCH_STRINGS)
    if file_found_strings:
        results.append({
            "file": full_path,
            "found_strings": file_found_strings
        })

# Handle both file and directory paths
if os.path.isfile(VOLUME_ROOT):
    scan_file(VOLUME_ROOT)
elif os.path.isdir(VOLUME_ROOT):
    for dirpath, _, filenames in os.walk(VOLUME_ROOT):
        for fname in filenames:
            full_path = os.path.join(dirpath, fname)
            scan_file(full_path)
else:
    print(f"WARNING: Path does not exist or is inaccessible: {VOLUME_ROOT}")

# ---- REPORT ----
print(f"Scanned {file_count} files ({error_count} unreadable)\n")
print(f"Searching for occurrences of {len(SEARCH_STRINGS)} strings: {', '.join(SEARCH_STRINGS)}\n")

if not results:
    print(" NO OCCURRENCES FOUND for any of the specified strings.\n")
else:
    for r in results:
        print(f" File: {r['file']}")
        for search_str_found, hits in r["found_strings"].items():
            print(f"  Found '{search_str_found}':")
            for h in hits:
                print(f"   Line {h['line']}: {h['match']}")
        print() # Add a blank line for readability between files

# ---- SAVE REPORT AS JSON FOR FURTHER ANALYSIS ----
# --- MODIFIED: Filename to reflect multiple string search ---
# Note: Creating a very long filename with all strings might be impractical.
# For simplicity, we'll just indicate "multiple_strings".
output_dir = os.path.dirname(VOLUME_ROOT) if os.path.isfile(VOLUME_ROOT) else VOLUME_ROOT
output_path = os.path.join(output_dir, f"scan_results_multiple_strings_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
try:
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Report saved to: {output_path}")
except Exception as e:
    print(f"Could not save report (check volume write permissions): {e}")


Scanned 362 files (0 unreadable)

Searching for occurrences of 5 strings: conf_scd_shared, initialize_abc_logger_objects, ingest_distribute_adhoc_job_trigger, ingest_distribute_job_group_trigger_miu.py, ingest_distribute_job_group_trigger_1.py

 File: /Volumes/xliidw_dev_lpl/application/adhoc_files/scanner utility/Xli_Wins/XLI_WINS_Extract1/scan_results_multiple_strings_20260922_111514.json
  Found 'initialize_abc_logger_objects':
   Line 5: initialize_abc_logger_objects
   Line 8: initialize_abc_logger_objects

 File: /Volumes/xliidw_dev_lpl/application/adhoc_files/scanner utility/Xli_Wins/XLI_WINS_Extract1/t_miu_hitrate.ipynb
  Found 'initialize_abc_logger_objects':
   Line 145: initialize_abc_logger_objects

Report saved to: /Volumes/xliidw_dev_lpl/application/adhoc_files/scanner utility/Xli_Wins/XLI_WINS_Extract1/scan_results_multiple_strings_20260922_111621.json
